# ViHSD Experiments — Colab workflow

ViHSD dùng `free_text`, `label_id`, PhoBERT và `max_length=128`. 
Validation macro F1 là metric chính; test chỉ dùng sau khi khóa finalist.

## 1. Mount Google Drive

Checkpoint, results và predictions được lưu vào Drive để tồn tại sau khi Colab runtime kết thúc. Hãy bật GPU và Internet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repository và cài dependencies

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

## 3. Xác thực Hugging Face và Weights & Biases

Tạo Colab Secrets `HF_TOKEN` và `WANDB_API_KEY`. Credentials được đọc tại runtime, không lưu trong notebook. Output được tách tại `ViHSD-MoE`.

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('ViHSD Colab environment configured.')

## 4. Kiểm tra dữ liệu và preprocessing

Dataset mặc định là `data/vihsd_segmented`, với split `train`, `validation`, `test`, text column `free_text` và label column `label_id`. 
Nếu chạy mô hình khác không phải PhoBERT, dùng `--set dataset.name='uitnlp/ViHSD'` để override config.

## 5. Training commands

Chạy smoke test trước. Sau đó chạy dense control, Transformer-MoE control hoặc finalist. Mỗi lệnh phải có `run_id` riêng; không ghi đè run cũ.

In [ ]:
# Smoke test
!python train.py --config configs/vihsd.yaml --smoke-test --run-id vihsd-smoke-s42

# Documented finalist — all layers, 4 experts, top-2, learnable residual scale.
# !python train.py --config configs/finalists/vihsd_best.yaml --no-smoke-test \
#   --run-id vihsd-moe-finalist-all12-e4-top2-s42 --set seed=42

## 6. Controlled ablations và repeated seeds

Các overlay trong `configs/stages/vihsd/` thay từng factor: placement, shared expert, residual scale, expert type hoặc initialization noise. Khi candidate ổn định, lặp seeds `42`, `43`, `44` và báo cáo mean ± standard deviation.

```bash
python train.py --config configs/finalists/vihsd_best.yaml --no-smoke-test \
  --run-id vihsd-moe-finalist-all12-e4-top2-s43 --set seed=43
```

Không coi một single-seed result là confirmed finding; xét cả validation macro F1, efficiency và routing behavior.

## 7. Evaluation command

Chỉ đánh giá test sau khi candidate/finalist được chọn bằng validation. Evaluation ghi metrics, predictions, confusion matrix và phân phối nhãn vào `results/<run-id>/`.

In [ ]:
# Thay run_id bằng run train thành công.
# !python evaluate.py --final-evaluation --config configs/finalists/vihsd_best.yaml --run-id vihsd-moe-finalist-all12-e4-top2-s42

# Debug checkpoint trực tiếp:
# !python evaluate.py --final-evaluation --config configs/finalists/vihsd_best.yaml \
#   --checkpoint checkpoints/vihsd-moe-finalist-all12-e4-top2-s42/transformer_moe_best.safetensors

## 8. Artifact, tracker và reference

Kiểm tra `resolved_config.yaml`, metadata, `run_metrics.json`, classification report, predictions và confusion matrix. 

| Thành phần | Giá trị |
|---|---|
| Dataset | `data/vihsd_segmented` |
| Input | `free_text`, single text |
| Label | `label_id` |
| Backbone chính | `vinai/phobert-base` |
| MoE control | layers `[8,9,10,11]`, 4 experts, top-1 |
| Finalist config | `configs/finalists/vihsd_best.yaml` |
| Selection metric | validation macro F1 |
| Seeds xác nhận | 42, 43, 44 |

